## Without web search

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from azure_openai_llm import create_azure_llm
from langchain.agents import create_agent
from langchain.messages import HumanMessage  

llm = create_azure_llm()

agent = create_agent(
    model=llm
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [3]:
agent.invoke(
    {"messages": [HumanMessage(content="who is current mayor of lahore?")]}
)['messages'][-1].content

'As of June 2024, the current mayor of Lahore is Mian Yasir Gillani.'

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)
response['messages'][-1].content

'My training includes information up until November 2023. If you have questions about events or developments after that date, I might not have the latest details. How can I assist you today?'

## Add web search tool

In [5]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of Lahore?")

{'query': 'Who is the current mayor of Lahore?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_Lahore',
   'title': 'Mayor of Lahore - Wikipedia',
   'content': 'Mayors of Lahore ; Mian Amir Mehmood, 2005, 2009 ; Administrator system implemented 2010–2016 ; Mubashar Javed, 2016, 2021 ; Administrator system implemented',
   'score': 0.8585411,
   'raw_content': None},
  {'url': 'https://www.dawn.com/news/1653640',
   'title': 'Lahore mayor assumes charge of office - Pakistan - DAWN.COM',
   'content': 'LAHORE: Retired Col Mubashir Javed of the PML-N assumed the charge of mayor office here on Saturday and the Punjab government formally',
   'score': 0.8377398,
   'raw_content': None},
  {'url': 'https://voicepk.net/2021/03/lahore-mayor-threatens-to-move-non-compliance-plea-against-punjab-govt-in-sc/',
   'title': 'Lahore mayor threatens to move non-compliance plea against ...',
   'content': 'Col Mubasher Javed,

In [6]:
agent = create_agent(
    llm,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of Lahore?")

response = agent.invoke(
    {"messages": [question]}
)

In [7]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of Lahore?', additional_kwargs={}, response_metadata={}, id='79f3eb57-307a-4207-9d00-019ac8316005'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 51, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DQoCYWjse5NaLwjsTiaOWsMEIYfbZ', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'saf

In [8]:
response['messages'][-1].content

'The current mayor of Lahore is Retired Col Mubashir Javed. He assumed charge of the mayor office recently.'

In [9]:
pprint(response["messages"][1].tool_calls)

[{'args': {'query': 'current mayor of Lahore'},
  'id': 'call_kfnh0AmoljdadWO7DteL8R4E',
  'name': 'web_search',
  'type': 'tool_call'}]


In [10]:
from langchain_core.tracers.context import tracing_v2_enabled

with tracing_v2_enabled() as ctx:
    question = HumanMessage(content="How is the weather today in Lahore?")

    response = agent.invoke(
        {"messages": [question]}
    )
    pprint(response['messages'][-1].content)
    url = ctx.get_run_url()
    print(f"View the trace at: {url}")

('The weather today in Lahore is overcast with a temperature of about 23.3°C '
 '(73.9°F). The humidity is around 61%, and there is a light northeast wind '
 'blowing at 3.6 kph (2.2 mph). The conditions are generally cloudy with about '
 '50% cloud cover.')
View the trace at: https://eu.smith.langchain.com/o/ed681c67-0bab-40c3-ad5a-fe23a6c3ddb4/projects/p/c01085aa-1fe5-450f-995e-347c849660ab/r/019d570c-6378-7b71-a4e2-cb7e569a588c?poll=true
